# 03 — Exploratory Data Analysis (EDA)
**NST DVA Capstone 2 · NASA Planetary Defense · Team SectionC_G-09**

## Objectives
- Understand distributions of key physical and orbital properties
- Identify patterns in close approach timing and velocity
- Compare PHAs vs non-PHAs on size, MOID, and orbital eccentricity
- Surface Tableau KPIs through visual exploration

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROCESSED_DIR = Path().resolve().parent / 'data' / 'processed'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

nea    = pd.read_csv(PROCESSED_DIR / 'nea_catalogue_clean.csv', low_memory=False)
pha    = pd.read_csv(PROCESSED_DIR / 'nea_hazardous_clean.csv', low_memory=False)
close  = pd.read_csv(PROCESSED_DIR / 'close_approaches_clean.csv')
future = pd.read_csv(PROCESSED_DIR / 'close_approaches_future_clean.csv')
close['close_approach_date']  = pd.to_datetime(close['close_approach_date'],  errors='coerce')
future['close_approach_date'] = pd.to_datetime(future['close_approach_date'], errors='coerce')
print(f'nea:{nea.shape}  pha:{pha.shape}  close:{close.shape}  future:{future.shape}')

## 3.1 — Absolute Magnitude & Diameter Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(nea['absolute_magnitude_h'].dropna(), bins=60, color='steelblue', edgecolor='white', lw=0.4)
axes[0].axvline(nea['absolute_magnitude_h'].median(), color='tomato', ls='--', label=f"Median: {nea['absolute_magnitude_h'].median():.1f}")
axes[0].set(title='Absolute Magnitude H Distribution', xlabel='H (lower = brighter/larger)', ylabel='Count')
axes[0].legend()
diam = nea['diameter_km'].dropna()
axes[1].hist(diam[diam < 50], bins=80, color='darkorange', edgecolor='white', lw=0.4)
axes[1].set(title='Diameter Distribution (km, <50 km)', xlabel='Diameter (km)', ylabel='Count')
plt.suptitle('NEA Physical Properties', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.2 — Orbit Class Breakdown

In [ ]:
col = 'orbit_class_label' if 'orbit_class_label' in nea.columns else 'class'
counts = nea[col].value_counts()
fig, ax = plt.subplots(figsize=(13, 5))
ax.barh(counts.index, counts.values, color=sns.color_palette('tab10', len(counts)))
ax.set(xlabel='Count', title='Asteroid Count by Orbit Class')
for bar in ax.patches:
    ax.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2, f'{int(bar.get_width()):,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3.3 — Orbital Eccentricity vs Semi-Major Axis (AU)

In [ ]:
samp = nea[['orbital_eccentricity','semi_major_axis_au','is_potentially_hazardous']].dropna()
samp = samp[samp['semi_major_axis_au'] < 5]
fig, ax = plt.subplots(figsize=(11, 6))
colors = samp['is_potentially_hazardous'].astype(str).map({'True':'crimson','False':'steelblue'})
ax.scatter(samp['semi_major_axis_au'], samp['orbital_eccentricity'], c=colors, alpha=0.3, s=8)
ax.axvline(1.0, color='gold', ls='--', lw=1)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='crimson',label='PHA'), Patch(color='steelblue',label='Non-PHA')])
ax.set(xlabel='Semi-Major Axis (AU)', ylabel='Orbital Eccentricity',
       title='Eccentricity vs Semi-Major Axis (gold line = 1 AU)')
plt.tight_layout()
plt.show()

## 3.4 — MOID: All NEAs vs PHAs

In [ ]:
moid = 'min_orbit_intersection_dist_au'
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(nea[moid].dropna().clip(upper=0.2), bins=80, alpha=0.5, color='steelblue', label='All NEAs')
ax.hist(pha[moid].dropna().clip(upper=0.2), bins=80, alpha=0.8, color='crimson', label='PHAs only')
ax.axvline(0.05, color='gold', ls='--', lw=1.5, label='PHA threshold (0.05 AU)')
ax.set(xlabel='MOID (AU)', ylabel='Count', title='MOID Distribution — All NEAs vs PHAs')
ax.legend()
plt.tight_layout()
plt.show()

## 3.5 — Annual Close Approach Count (2015–2035)

In [ ]:
if 'approach_year' not in close.columns:
    close['approach_year'] = close['close_approach_date'].dt.year
yearly = close.groupby('approach_year').size().reset_index(name='count')
fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(yearly['approach_year'], yearly['count'], alpha=0.25, color='steelblue')
ax.plot(yearly['approach_year'], yearly['count'], color='steelblue', lw=2, marker='o', ms=4)
ax.axvline(2025, color='tomato', ls='--', lw=1.5, label='Present (2025)')
ax.set(xlabel='Year', ylabel='Close Approaches', title='Annual Close Approach Count (2015–2035)')
ax.legend()
plt.tight_layout()
plt.show()

## 3.6 — Velocity Distribution & Speed Categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vel = close['velocity_km_s'].dropna()
vel = vel[vel < vel.quantile(0.99)]
axes[0].hist(vel, bins=70, color='seagreen', edgecolor='white', lw=0.3)
axes[0].axvline(vel.median(), color='red', ls='--', label=f'Median: {vel.median():.1f} km/s')
axes[0].set(title='Approach Velocity (km/s)', xlabel='Velocity (km/s)', ylabel='Count')
axes[0].legend()
if 'speed_category' in close.columns:
    cat = close['speed_category'].value_counts()
    axes[1].pie(cat.values, labels=cat.index, autopct='%1.1f%%',
                colors=sns.color_palette('Set2', len(cat)), startangle=140)
    axes[1].set_title('Speed Category Distribution')
plt.tight_layout()
plt.show()

## 3.7 — Future Approaches: Distance vs Velocity

In [ ]:
f = future[['distance_lunar_distances','velocity_km_s','absolute_magnitude']].dropna()
f = f[f['distance_lunar_distances'] < 100]
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(f['distance_lunar_distances'], f['velocity_km_s'],
                c=f['absolute_magnitude'], cmap='plasma_r', alpha=0.5, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Absolute Magnitude H (lower = larger)')
ax.set(xlabel='Distance (Lunar Distances)', ylabel='Velocity (km/s)',
       title='Future Close Approaches (≥2025) — Distance vs Velocity')
plt.tight_layout()
plt.show()

## 3.8 — Risk Tier Breakdown

In [ ]:
if 'risk_tier' in nea.columns:
    tier = nea['risk_tier'].value_counts()
    cmap = {'Critical':'#d62728','High':'#ff7f0e','Moderate':'#ffdd57','Low':'#2ca02c'}
    bar_colors = [cmap.get(t,'#aec7e8') for t in tier.index]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(tier.index, tier.values, color=bar_colors, edgecolor='white')
    ax.set(title='Asteroid Risk Tier Distribution', xlabel='Risk Tier', ylabel='Count')
    for i, v in enumerate(tier.values):
        ax.text(i, v+10, f'{v:,}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('risk_tier column not found — run pipeline.py first')

## 3.9 — EDA Summary
| Finding | Insight |
|---|---|
| Most NEAs are small (H>18) | Few city-killer+ objects in catalogue |
| PHAs cluster at MOID < 0.05 AU | MOID is the primary hazard discriminator |
| Apollos dominate orbit classes | Most Earth-crossing NEAs are Apollos |
| Approach velocity peaks 10–25 km/s | Aligns with known NEA encounter speeds |

**Next → Notebook 04: Statistical Analysis**